# Shift-Scan — Final Experiments (Google Drive-ready)
This notebook is configured for the two folders visible in your Google Drive:
- `MyDrive/shift_scan_advanced_project`
- `MyDrive/shift_scan_recovered_data`

It discovers recovered JSONL files directly inside the recovered-data folder (including nested folders), keeps the recovered records unchanged, and places fresh outputs under the project folder. It does not assume you have a ZIP file.

**Research integrity:** old results are context, not fresh executions. Missing checkpoints cannot be recreated from metric logs. Only claim results produced by runs completed in this notebook.

## 1. Set GPU runtime and mount Drive
In Colab, choose **Runtime → Change runtime type → GPU**, then run this cell.

In [ ]:
import os, sys, json, csv, time, subprocess, platform, random, shutil
from pathlib import Path
import numpy as np
import pandas as pd

from google.colab import drive
drive.mount('/content/drive')

MYDRIVE = Path('/content/drive/MyDrive')
PROJECT_ROOT = MYDRIVE / 'shift_scan_advanced_project'
RECOVERED_ROOT = MYDRIVE / 'shift_scan_recovered_data'
RUNS_DIR = PROJECT_ROOT / 'results' / 'final_controlled'
RUNS_DIR.mkdir(parents=True, exist_ok=True)

print('Project folder:', PROJECT_ROOT, '| exists:', PROJECT_ROOT.exists())
print('Recovered-data folder:', RECOVERED_ROOT, '| exists:', RECOVERED_ROOT.exists())
try:
    import torch
    print('Torch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
        print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
except Exception as e:
    print('Torch check issue:', repr(e))

## 2. Check your actual Drive folder contents
This prints the first levels so you can confirm Colab sees the same folders as your screenshot.

In [ ]:
for root in [PROJECT_ROOT, RECOVERED_ROOT]:
    print('\\n---', root, '---')
    if root.exists():
        for p in sorted(root.iterdir()):
            print(('DIR  ' if p.is_dir() else 'FILE '), p.name)
    else:
        print('NOT FOUND — check folder spelling/location in Drive.')

## 3. Find recovered data files automatically
Looks for `runs.jsonl`, `shift_scan_runs_recovered.jsonl`, `supplemental_metrics.jsonl`, CSV files and README provenance notes. Does not edit the source files.

In [ ]:
if not RECOVERED_ROOT.exists():
    raise FileNotFoundError(f'Cannot find {RECOVERED_ROOT}. Check your Drive folder name.')

all_files = [p for p in RECOVERED_ROOT.rglob('*') if p.is_file()]
print('Files found:', len(all_files))
for p in all_files[:150]:
    print(p.relative_to(RECOVERED_ROOT))

run_candidates = [p for p in all_files if p.name in {'runs.jsonl','shift_scan_runs_recovered.jsonl'}]
supp_candidates = [p for p in all_files if p.name == 'supplemental_metrics.jsonl']
csv_candidates = [p for p in all_files if p.suffix.lower() == '.csv']
readme_candidates = [p for p in all_files if 'readme' in p.name.lower()]

print('\\nRun JSONL candidates:', run_candidates)
print('Supplemental JSONL candidates:', supp_candidates)
print('CSV files:', csv_candidates)

## 4. Load the recovered canonical records
If both `runs.jsonl` and the standalone recovered file are present, select the original nested `runs.jsonl` first, otherwise select the standalone file. Inspect the candidates if the printout shows a different arrangement.

In [ ]:
def read_jsonl(path):
    rows=[]
    if path is None: return rows
    with open(path, 'r', encoding='utf-8') as f:
        for line_no, line in enumerate(f, 1):
            line=line.strip()
            if not line: continue
            try: rows.append(json.loads(line))
            except Exception as e: print(f'Malformed line {line_no} in {path}: {e}')
    return rows

def prefer(path_list):
    if not path_list: return None
    # Prefer archive's canonical runs.jsonl, then recovered standalone filename.
    path_list=sorted(path_list, key=lambda p: (p.name != 'runs.jsonl', len(str(p))))
    return path_list[0]

RUNS_JSONL = prefer(run_candidates)
SUPP_JSONL = prefer(supp_candidates)
recovered_runs = read_jsonl(RUNS_JSONL)
supplemental = read_jsonl(SUPP_JSONL)

print('Canonical source:', RUNS_JSONL)
print('Canonical records:', len(recovered_runs))
print('Supplemental source:', SUPP_JSONL)
print('Supplemental records:', len(supplemental))

runs_df = pd.json_normalize(recovered_runs)
display(runs_df.head(12))

## 5. Prior experiment summary (recovered; not fresh runs)
Budgets differ between these rows. Keep them separately labeled; do not mix them into one matched-seed aggregate.

In [ ]:
historical = pd.DataFrame([
 {'experiment':'FP student','budget':'4M','seeds':'3','PPL':'23.0868 / 23.1094 / 23.0948','provenance':'recovered'},
 {'experiment':'Ternary absmean','budget':'4M','seeds':'3','PPL':'60.8882 / 70.9303 / 55.3509','provenance':'recovered'},
 {'experiment':'Ternary TWN','budget':'4M','seeds':'3','PPL':'44.0985 / 43.2672 / 45.5476','provenance':'recovered'},
 {'experiment':'Ternary act-aware','budget':'4M','seeds':'3','PPL':'48.5344 / 50.0394 / 50.4829','provenance':'recovered'},
 {'experiment':'v2_best','budget':'4M','seeds':'3','PPL':'31.79395 / 31.76474 / 31.94346','provenance':'recovered'},
 {'experiment':'v2_best nearest','budget':'4M','seeds':'1','PPL':'31.81958','provenance':'recovered'},
 {'experiment':'v2_best dyadic2','budget':'4M','seeds':'1','PPL':'31.82718','provenance':'recovered'},
 {'experiment':'v2_best_long','budget':'12M','seeds':'1','PPL':'29.28271','provenance':'recovered'},
 {'experiment':'HARPQ','budget':'~20M','seeds':'0','PPL':'133.9073','provenance':'distinct protocol'},
])
display(historical)

## 6. Lock matched ablation design
Primary factors: projection precision (FP16 vs ternary TWN) and decay (original FP vs power-of-two). Keep dataset stream, tokenizer, initialization, optimizer, schedule, token budget, and evaluation identical. Pilot budget is 2M tokens; scaling to 4M/12M/20M is separate and only after pilot validation.

In [ ]:
ABLATIONS = [
    {'code':'A0','projections':'FP16','decay':'original_FP','meaning':'reference baseline'},
    {'code':'A1','projections':'ternary_TWN','decay':'original_FP','meaning':'projection quantization only'},
    {'code':'A2','projections':'FP16','decay':'power_of_two','meaning':'decay parameterization only'},
    {'code':'A3','projections':'ternary_TWN','decay':'power_of_two','meaning':'combined method'},
]
SEEDS = [0, 1, 2]
TARGET_TOKENS = 2_000_000
SEQ_LEN = 512
HORIZON = 128
PROTOCOL_ID = f'matched_{TARGET_TOKENS//1_000_000}M_seq{SEQ_LEN}_seeds012_v1'
display(pd.DataFrame(ABLATIONS))
print('Protocol ID:', PROTOCOL_ID)

## 7. Inspect project code/checkpoints
This lists available scripts and checkpoint paths to help you select the correct actual trainer. The notebook deliberately will not guess CLI arguments.

In [ ]:
if PROJECT_ROOT.exists():
    pyfiles = sorted(p for p in PROJECT_ROOT.rglob('*.py'))
    print('Python scripts:', len(pyfiles))
    for p in pyfiles[:120]: print(p.relative_to(PROJECT_ROOT))
    ckpts = [p for p in PROJECT_ROOT.rglob('*') if p.is_file() and p.suffix.lower() in {'.pt','.pth','.ckpt','.safetensors'}]
    print('\\nCheckpoint count:', len(ckpts))
    for p in ckpts[:50]: print(p.relative_to(PROJECT_ROOT))
else:
    print('Project root missing.')

## 8. Configure verified training CLI
Review scripts in Section 7. Replace the blank command with your real project trainer command. Template variables allowed by this notebook: `{seed}`, `{ablation}`, `{tokens}`, `{seq_len}`, `{out}`. This stays disabled until you explicitly enable the execution cell.

In [ ]:
# Example ONLY. Replace with the actual command used by your repository.
COMMAND_TEMPLATE = ''

def make_command(ablation, seed, out_dir):
    if not COMMAND_TEMPLATE.strip():
        raise RuntimeError('Set COMMAND_TEMPLATE to the verified project training command first.')
    return COMMAND_TEMPLATE.format(seed=seed, ablation=ablation['code'],
                                   tokens=TARGET_TOKENS, seq_len=SEQ_LEN,
                                   out=str(out_dir))

print('Command configured:', bool(COMMAND_TEMPLATE.strip()))

## 9. Run the matched matrix
Run only after verifying command/config maps A0–A3 correctly. Each run writes metadata and a log to Drive. Failed runs preserve return code and log.

In [ ]:
RUN_EXPERIMENTS = False  # change to True only after reviewing command

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    except Exception as e: print('Seed warning:', e)

if RUN_EXPERIMENTS:
    for ab in ABLATIONS:
        for seed in SEEDS:
            run_id=f'{PROTOCOL_ID}_{ab["code"]}_seed{seed}'
            out_dir=RUNS_DIR/run_id
            out_dir.mkdir(parents=True, exist_ok=True)
            cmd=make_command(ab, seed, out_dir)
            meta={'run_id':run_id,'protocol_id':PROTOCOL_ID,'ablation':ab,
                  'seed':seed,'target_tokens':TARGET_TOKENS,'seq_len':SEQ_LEN,
                  'command':cmd,'started_unix':time.time()}
            with open(out_dir/'run_metadata.json','w') as f: json.dump(meta,f,indent=2)
            set_seed(seed)
            with open(out_dir/'stdout_stderr.log','w') as log:
                proc=subprocess.run(cmd,shell=True,cwd=str(PROJECT_ROOT),
                                   stdout=log,stderr=subprocess.STDOUT)
            meta.update({'return_code':proc.returncode,'finished_unix':time.time()})
            with open(out_dir/'run_metadata.json','w') as f: json.dump(meta,f,indent=2)
            print(run_id,'return code:',proc.returncode,'log:',out_dir/'stdout_stderr.log')
else:
    print('Execution is OFF. Configure COMMAND_TEMPLATE and inspect it, then set RUN_EXPERIMENTS=True.')

## 10. Evaluation metrics template
Populate this from real evaluator output. Keep missing metrics empty; never infer them.

In [ ]:
METRIC_FIELDS=['run_id','protocol_id','ablation','seed','tokens_actual','seq_len',
 'fineweb_eval_ce','fineweb_eval_ppl','teacher_eval_ppl','state_mse','retention_error',
 'runtime_seconds','peak_vram_gib','checkpoint_path','status','notes']
METRICS_CSV=RUNS_DIR/'metrics_template.csv'
if not METRICS_CSV.exists():
    pd.DataFrame(columns=METRIC_FIELDS).to_csv(METRICS_CSV,index=False)
metrics=pd.read_csv(METRICS_CSV)
print('Metrics CSV:', METRICS_CSV)
display(metrics.head())

## 11. Aggregate only matched protocol results

In [ ]:
if len(metrics):
    metrics['fineweb_eval_ppl']=pd.to_numeric(metrics['fineweb_eval_ppl'],errors='coerce')
    matched=metrics[metrics['protocol_id']==PROTOCOL_ID]
    display(matched.groupby('ablation')['fineweb_eval_ppl'].agg(['count','mean','std']).reset_index())
else:
    print('No evaluation rows yet — this is expected before actual runs finish.')

## 12. Long-context stability follow-up
After pilot completes, evaluate existing checkpoints at supported lengths, e.g. 512/1024/2048. Record PPL by length, state norms, finite-value checks, runtime and memory. Only configure a command if your evaluator supports these options.

In [ ]:
LONG_CONTEXT_LENGTHS=[512,1024,2048]
LONG_EVAL_COMMAND_TEMPLATE=''
print('Long-context evaluator command configured:', bool(LONG_EVAL_COMMAND_TEMPLATE.strip()))

## 13. Integrity checks

In [ ]:
def integrity_check(df):
    required={'run_id','protocol_id','ablation','seed','tokens_actual','fineweb_eval_ppl','status'}
    missing=required-set(df.columns)
    if missing: raise ValueError('Missing columns: '+str(sorted(missing)))
    print('duplicate run IDs:', int(df.run_id.duplicated().sum()))
    print('rows not marked success:', int((df.status!='success').sum()))
    print('missing PPL:', int(df.fineweb_eval_ppl.isna().sum()))
    print('actual tokens range:', df.tokens_actual.min(), df.tokens_actual.max())

if len(metrics): integrity_check(metrics)

## 14. Export project data and logs
Create a ZIP in the project Drive folder, containing the recovered data references and new artifacts separately.

In [ ]:
EXPORT_DIR=Path('/content/shift_scan_final_export')
EXPORT_DIR.mkdir(exist_ok=True)
if RUNS_JSONL and Path(RUNS_JSONL).exists(): shutil.copy2(RUNS_JSONL,EXPORT_DIR/'recovered_original_runs.jsonl')
if SUPP_JSONL and Path(SUPP_JSONL).exists(): shutil.copy2(SUPP_JSONL,EXPORT_DIR/'recovered_supplemental_metrics.jsonl')
if METRICS_CSV.exists(): shutil.copy2(METRICS_CSV,EXPORT_DIR/'new_experiment_metrics.csv')
if RUNS_DIR.exists(): shutil.copytree(RUNS_DIR,EXPORT_DIR/'new_run_artifacts',dirs_exist_ok=True)
zip_path=PROJECT_ROOT/'shift_scan_final_export.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as zf:
    for p in EXPORT_DIR.rglob('*'):
        if p.is_file(): zf.write(p,p.relative_to(EXPORT_DIR.parent))
print('Export written to Drive:',zip_path)

## 15. Final reporting boundaries
- Report A0–A3 as a matched comparison only if all same-protocol seeds ran.
- Report per-seed values, mean and sample standard deviation.
- Separate pilot (2M), existing 4M, 12M, and ~20M results.
- Efficiency requires measured latency/tokens-per-second/VRAM on actual hardware. Analytical multiplication estimates or the slow prototype kernel do not demonstrate speedup.
- State the limitations: checkpoint availability, hardware/runtime constraints, architecture scope and incomplete evaluations.